In [1]:
# %%
import pickle
import pandas as pd
from IPython.display import display, Markdown

# ==========================================
# ⏱️ 量化时光机：机构级每日战报 (升级版)
# ==========================================
TARGET_DATE = "2026-05-12"  # 你可以随时改成其他交易日
report_file = f"Daily_Reports/Report_{TARGET_DATE}.pkl"

try:
    with open(report_file, 'rb') as f:
        snap = pickle.load(f)

    # ✨ 核心修复：提取最强与次强因子，建立准确的内存变量映射
    king_f = snap.get('king_factor', 'Momentum')  # 增加容错，兜底默认主线
    sub_factor = snap.get('sub_factor', '未知')

    # 标题头
    display(Markdown(f"# 📊 宽客实战日报 | 交易日: {snap['date']}"))
    display(Markdown(
        f"### 👑 市场绝对主线: **`{snap['king_factor']}`** | 🥈 次强辅线: **`{sub_factor}`**"))
    display(Markdown("---"))

    # ---------------------------------------------------------
    # 模块 1：OLS 市场风向归因
    # ---------------------------------------------------------
    r2_fac = snap['ols_report']['r2_factors'] * 100
    r2_ind = snap['ols_report']['r2_ind'] * 100
    display(Markdown(
        f"#### 📡 1.1 市场微观结构探测 (解释力 - 因子: `{r2_fac:.2f}%` | 行业: `{r2_ind:.2f}%`)"))

    style_ols = snap['ols_report']['stats_df'].style.map(
        lambda x: 'color: red; font-weight: bold' if isinstance(x, str) and '追捧' in x else
                  ('color: green; font-weight: bold' if isinstance(x, str)
                   and '抛售' in x else ''),
        subset=['风向状态']
    ).background_gradient(subset=['Beta系数'], cmap='coolwarm', vmin=-0.05, vmax=0.05)
    display(style_ols)

    display(Markdown(f"#### 🏆 1.2 因子全维度强度对比"))
    king_stats_styled = snap['king_stats'].style.background_gradient(
        subset=['Beta (强度)'], cmap='RdYlGn'
    ).map(
        lambda x: 'font-weight: bold; color: #FF4500' if isinstance(
            x, (int, float)) and abs(x) > 1.96 else '',
        subset=['T值 (显著性)']
    )
    display(king_stats_styled)
    
    # ---------------------------------------------------------
    # 模块 2：【全新】板块时序动量与资金跃迁监控
    # ---------------------------------------------------------
    if 'sector_rank' in snap:
        display(Markdown("#### 🔄 2. 核心主舞台轮动透视 (已过滤低成交量噪音)"))
        rank_df = snap['sector_rank']

        # 提取排名飙升（抢筹）和暴跌（撤离）的板块
        soaring = rank_df.sort_values(by='Change', ascending=False).head(5)
        diving = rank_df.sort_values(by='Change', ascending=True).head(5)

        # ✨ 修复逻辑：格式化资金量（单位：亿元），让资金关注度一目了然
        for df in [soaring, diving]:
            if 'T0_Amount' in df.columns:
                df['成交额(亿)'] = (df['T0_Amount'] /
                                1e8).apply(lambda x: f"{x:.1f}")

        soaring_show = soaring[['T0_Rank', 'Change', '成交额(亿)']].rename(
            columns={'T0_Rank': '当前排名', 'Change': '跃升(位)'}
        )
        diving_show = diving[['T0_Rank', 'Change', '成交额(亿)']].rename(
            columns={'T0_Rank': '当前排名', 'Change': '下跌(位)'}
        )

        # 美化并排展示
        display(pd.concat([soaring_show, diving_show],
                axis=1, keys=['🚀 主力抢筹飙升', '📉 主力退潮撤离']))
        display(Markdown("---"))

    # ---------------------------------------------------------
    # 模块 3：截面强势板块下钻与龙头画像
    # ---------------------------------------------------------
    display(Markdown("#### 🎯 3. 截面强势板块下钻与龙头画像"))
    col1, col2 = snap['drill_report']['top_industries'], snap['drill_report']['bottom_industries']
    display(pd.concat([col1.set_index('板块名称'), col2.set_index(
        '板块名称')], axis=1, keys=['🔥 领涨板块', '❄️ 领跌板块']))

    display(Markdown("**【强势板块内部资金画像】**"))
    display(snap['drill_report']['portraits_df'].style.set_properties(
        **{'text-align': 'left'}))
    display(Markdown("---"))

    # ---------------------------------------------------------
    # 模块 4：【修复重点】全市场(ZZ800) 终极猎物池
    # ---------------------------------------------------------
    if 'zz800_top100' in snap:
        display(Markdown(
            f"#### 🌐 4. 全市场(ZZ800) 复合多因子 Top 20 (70% `{king_f}` + 30% `{sub_factor}`)"))

        df_zz800 = snap['zz800_top100'].copy()

        # ✨ 修复逻辑：显式定义要展示的列，确保 'name' 在前排
        # 如果 'name' 缺失（显示为 NaN），则填充为 '未知'
        if 'name' in df_zz800.columns:
            df_zz800['name'] = df_zz800['name'].fillna('未匹配名称')
        else:
            df_zz800['name'] = '缺少名称列'

        # 定义展示列清单
        show_cols = ['code', 'name', 'Industry',
                     'Composite_Score', king_f, sub_factor]
        # 过滤掉不存在于 DataFrame 中的列，防止报错
        actual_show_cols = [c for c in show_cols if c in df_zz800.columns]

        # 渲染前 20 名
        top_20_view = df_zz800.head(20)[actual_show_cols]

        display(top_20_view.style.background_gradient(
            subset=['Composite_Score'], cmap='YlOrRd'
        ).format({
            'Composite_Score': "{:.2f}",
            king_f: "{:.2f}" if king_f in top_20_view.columns else "{}",
            sub_factor: "{:.2f}" if sub_factor in top_20_view.columns else "{}"
        }))
        display(Markdown("---"))

    # ---------------------------------------------------------
    # 模块 5：个人底仓洗牌监控
    # ---------------------------------------------------------
    display(
        Markdown(f"#### 🔫 5. 个人底仓洗牌池 (按主线 **`{snap['king_factor']}`** 降序排列)"))

    # 动态组装你想看的因子列，确保不重复
    display_cols = ['code', 'name', snap['king_factor'],
                    sub_factor, 'Momentum', 'Liquidity', 'Size']
    display_cols = list(dict.fromkeys(display_cols))  # 去重，保持顺序

    # 只取存在的列进行展示
    actual_cols = [c for c in display_cols if c in snap['top_picks'].columns]
    top_picks = snap['top_picks'].head(20)[actual_cols]

    display(top_picks.style.background_gradient(
        subset=[snap['king_factor']], cmap='YlOrRd'))


except FileNotFoundError:
    print(f"❌ 找不到 {TARGET_DATE} 的战报！可能当天未生成或日期错误。")
# %%

# 📊 宽客实战日报 | 交易日: 2026-05-12

### 👑 市场绝对主线: **`Momentum`** | 🥈 次强辅线: **`Short_Rev`**

---

#### 📡 1.1 市场微观结构探测 (解释力 - 因子: `18.50%` | 行业: `22.43%`)

,因子 (Factor),Beta系数,T值 (显著性),风向状态
0,Momentum,39.634000,5.210000,追捧 ★
1,Short_Rev,27.824000,3.810000,追捧 ★
2,Low_Vol,-27.257000,-3.460000,抛售 ★
3,Liquidity,48.455000,6.500000,追捧 ★
4,Size,-9.552000,-1.230000,抛售
5,Value_BP,-28.970000,-2.710000,抛售 ★
6,Mom_Sharpe,18.921000,2.370000,追捧 ★
7,Vol_Price_Corr,7.863000,1.060000,追捧
8,Amihud,-15.069000,-1.900000,抛售


...

#### 🌐 4. 全市场(ZZ800) 复合多因子 Top 20 (70% `Momentum` + 30% `Short_Rev`)

,code,name,Industry,Composite_Score,Momentum,Short_Rev
0,sz.300390,天华新能,C39计算机、通信和其他电子设备制造业,2.71,3.23,1.52
1,sz.300408,三环集团,C39计算机、通信和其他电子设备制造业,2.19,3.13,-0.02
2,sz.000967,盈峰环境,C35专用设备制造业,1.99,2.79,0.10
3,sh.688220,翱捷科技,C39计算机、通信和其他电子设备制造业,1.98,3.03,-0.46
4,sh.688702,盛科通信,I65软件和信息技术服务业,1.87,2.05,1.45
5,sz.301358,湖南裕能,C39计算机、通信和其他电子设备制造业,1.83,2.36,0.61
6,sh.600208,衢州发展,K70房地产业,1.78,3.23,-1.59
7,sz.002384,东山精密,C39计算机、通信和其他电子设备制造业,1.63,3.23,-2.08
8,sh.688041,海光信息,C39计算机、通信和其他电子设备制造业,1.62,3.23,-2.13
9,sh.688037,芯源微,C35专用设备制造业,1.62,3.23,-2.14


---

#### 🔫 5. 个人底仓洗牌池 (按主线 **`Momentum`** 降序排列)

,code,name,Momentum,Short_Rev,Liquidity,Size
0,688041,海光信息,3.225982,-2.128009,-0.239780,3.004812
1,688008,澜起科技,3.225982,-3.330880,1.805744,1.666955
2,300308,中际旭创,2.873028,-2.247515,0.273352,3.107111
3,688099,晶晨股份,2.405439,-0.564990,1.168344,-0.183490
4,688017,绿的谐波,2.201538,-2.739371,0.987128,-0.164897
5,688347,华虹公司,1.980046,-1.573307,1.914740,0.187985
6,2466,天齐锂业,1.650273,1.328308,2.244957,0.889978
7,688120,华海清科,1.610301,-2.301816,-0.138432,0.409746
8,338,潍柴动力,1.380907,-0.282806,0.171486,1.280428
9,600150,中国船舶,1.366236,0.822553,-0.185732,2.011665
